# Step 3: Retrieval + end-to-end fact-checker
At demo time nobody hands the model an abstract. So we need to **find** the evidence first.

**Pipeline:** claim → search a pool of abstracts → top-5 passages → verdict model (v2) on each → combine into one verdict.

**This notebook**
1. Builds the abstract pool: full SciFact corpus (5,183) + full HealthVer corpus (322).
2. Builds two searchers: **BM25** (keyword match) and **dense** (embedding similarity, `bge-small`), plus a **hybrid** that fuses both.
3. Measures **recall@k**: how often the right abstract is in the top k.
4. Runs the whole pipeline end to end and reports macro-F1 *without* being handed the gold abstract.
5. Saves the pool + embeddings for the demo.

**Runtime → Change runtime type → T4 GPU** before running. Approve the Drive pop-up.

In [ ]:
%pip install -q "datasets==3.6.0" "transformers>=4.46,<5" sentence-transformers rank_bm25 accelerate sentencepiece protobuf scikit-learn pandas pyarrow

In [ ]:
import os, re, json, collections, numpy as np, pandas as pd, torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/veritas"
except ImportError:                       # not on Colab (local test only)
    DRIVE = "veritas_local"
OUT = f"{DRIVE}/retrieval"; os.makedirs(OUT, exist_ok=True)
MODEL_DIR = f"{DRIVE}/verdict_model_v2"
EMB_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
TOPK = 5                    # passages handed to the verdict model
KS = (1, 5, 10)             # recall@k reported
DEV = "cuda" if torch.cuda.is_available() else "cpu"
LABELS = ["SUPPORTS", "REFUTES", "NEI"]
print("device:", DEV, "| model dir exists:", os.path.isdir(MODEL_DIR))

## 1. The abstract pool
Each document has a `title` and an `abstract`.
- **Searching** uses `title + abstract` (more words to match).
- **The verdict model** is shown the `abstract` only, because that is what it was trained on.

Both datasets come straight from their full corpora, not from the training table (which only held abstracts attached to a claim).

In [ ]:
sf_corpus = load_dataset("allenai/scifact", "corpus", trust_remote_code=True)["train"]
hv_all = load_dataset("dwadden/healthver_entailment", trust_remote_code=True)

docs = {}
for d in sf_corpus:
    docs[f"sf_{d['doc_id']}"] = dict(source="scifact", title=d["title"] or "", text=" ".join(d["abstract"]).strip())
for sp in hv_all:
    for ex in hv_all[sp]:
        docs.setdefault(f"hv_{ex['abstract_id']}", dict(source="healthver", title=ex["title"] or "",
                                                         text=" ".join(ex["abstract"]).strip()))
corpus = pd.DataFrame([dict(uid=u, **v) for u, v in docs.items()])
corpus["search_text"] = corpus["title"] + ". " + corpus["text"]
print(corpus.source.value_counts().to_dict(), "| total:", len(corpus))
print("median abstract words:", int(corpus.text.str.split().str.len().median()))

## 2. Evaluation claims
Three sets, all *claims the verdict model never trained on*:
SciFact validation, HealthVer validation, HealthVer test.

- **Gold documents**: the abstracts that actually contain evidence for the claim.
- **Gold label**: SUPPORTS / REFUTES from that evidence, or NEI if the claim has no evidence anywhere.

Claims with no gold document can't be scored for retrieval, but they **are** included in the end-to-end verdict score (the model should say NEI).

In [ ]:
def build_claims(rows):
    # rows: list of (claim_id, claim_text, doc_uid_or_None, label_name_or_None)
    by = collections.OrderedDict()
    for cid, claim, uid, lab in rows:
        e = by.setdefault(cid, dict(claim=claim.strip(), gold=set(), labs=[]))
        if uid is not None and lab in ("SUPPORTS", "REFUTES"):
            e["gold"].add(uid); e["labs"].append(lab)
    out = []
    for e in by.values():
        lab = collections.Counter(e["labs"]).most_common(1)[0][0] if e["labs"] else "NEI"
        out.append(dict(claim=e["claim"], gold=e["gold"], label=LABELS.index(lab)))
    return out

sf = load_dataset("allenai/scifact", "claims", trust_remote_code=True)["validation"]
m = {"SUPPORT": "SUPPORTS", "CONTRADICT": "REFUTES"}
sf_rows = [(ex["id"], ex["claim"], f"sf_{ex['evidence_doc_id']}" if ex["evidence_label"] else None, m.get(ex["evidence_label"])) for ex in sf]

hm = {"SUPPORT": "SUPPORTS", "CONTRADICT": "REFUTES", "NEI": "NEI"}
def hv_rows(split):
    return [(ex["claim_id"], ex["claim"], f"hv_{ex['abstract_id']}", hm.get(ex["verdict"])) for ex in hv_all[split]]

EVAL = {"SciFact val": build_claims(sf_rows),
        "HealthVer val": build_claims(hv_rows("validation")),
        "HealthVer test": build_claims(hv_rows("test"))}
for k, v in EVAL.items():
    n_gold = sum(bool(c["gold"]) for c in v)
    print(f"{k:15s} claims: {len(v):4d} | with gold evidence: {n_gold:4d} | NEI-only: {len(v) - n_gold:4d}")

## 3. Two searchers
- **BM25**: scores documents by shared rare words. Great for exact terms ("remdesivir"), blind to paraphrase.
- **Dense**: turns claim and abstract into vectors; nearby vectors = similar meaning. Handles paraphrase, can miss exact terms.
- **Hybrid (RRF)**: each searcher ranks the pool; documents ranked high by both win. Fusion score = Σ 1 / (60 + rank).

In [ ]:
tok_re = re.compile(r"[a-z0-9]+")
STOP = set("a an and are as at be by for from has have in is it its of on or that the this to was were will with".split())
def toks(t): return [w for w in tok_re.findall(t.lower()) if w not in STOP]

bm25 = BM25Okapi([toks(t) for t in corpus.search_text])

st = SentenceTransformer(EMB_MODEL, device=DEV); st.max_seq_length = 256
emb_path = f"{OUT}/corpus_emb.npy"
if os.path.exists(emb_path) and len(np.load(emb_path, mmap_mode="r")) == len(corpus):
    D = np.load(emb_path)
else:
    D = st.encode(corpus.search_text.tolist(), batch_size=64, normalize_embeddings=True,
                  show_progress_bar=True, convert_to_numpy=True).astype("float32")
print("doc embeddings:", D.shape)

def rank_bm25(queries, n=100):
    return [np.argsort(-bm25.get_scores(toks(q)))[:n] for q in queries]

def rank_dense(queries, n=100):
    Q = st.encode([QUERY_PREFIX + q for q in queries], batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
    S = Q @ D.T
    return [np.argsort(-s)[:n] for s in S]

def fuse(r1, r2, n=100, k=60):
    out = []
    for a, b in zip(r1, r2):
        sc = collections.defaultdict(float)
        for r in (a, b):
            for rank, i in enumerate(r): sc[i] += 1.0 / (k + rank + 1)
        out.append(np.array(sorted(sc, key=sc.get, reverse=True)[:n]))
    return out

uid_of = corpus.uid.values

## 4. Recall@k: is the right abstract in the top k?

In [ ]:
RANKS = {}   # (set name) -> dict(method -> list of rankings)
rows = []
for name, claims in EVAL.items():
    qs = [c["claim"] for c in claims]
    rb, rd = rank_bm25(qs), rank_dense(qs)
    RANKS[name] = {"BM25": rb, "Dense": rd, "Hybrid": fuse(rb, rd)}
    has_gold = [i for i, c in enumerate(claims) if c["gold"]]
    for meth, R in RANKS[name].items():
        rec = {k: np.mean([bool(set(uid_of[R[i][:k]]) & claims[i]["gold"]) for i in has_gold]) for k in KS}
        rows.append(dict(set=name, method=meth, **{f"recall@{k}": round(v, 3) for k, v in rec.items()}))
print(pd.DataFrame(rows).to_string(index=False))

## 5. End to end
For every claim: take the hybrid top-5, score each (claim, abstract) pair with the verdict model, then combine:
- `s` = highest SUPPORTS probability across the 5 passages, `r` = highest REFUTES probability
- evidence strength = `max(s, r)`
- if strength < **τ** → **NEI**; otherwise the larger of `s`, `r` wins

τ is swept below. Choosing it properly (calibration) is step 5.

In [ ]:
vtok = AutoTokenizer.from_pretrained(MODEL_DIR)
vmodel = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEV).eval()
if DEV == "cuda": vmodel = vmodel.half()

@torch.no_grad()
def verdict_probs(claim_doc_pairs, bs=32, max_len=384):
    out = []
    for i in range(0, len(claim_doc_pairs), bs):
        b = claim_doc_pairs[i:i + bs]
        enc = vtok([c for c, _ in b], [d for _, d in b], truncation="only_second", max_length=max_len,
                   padding=True, return_tensors="pt").to(DEV)
        out.append(torch.softmax(vmodel(**enc).logits.float(), -1).cpu().numpy())
    return np.concatenate(out)

def claim_scores(claims, rankings, topk=TOPK):
    pairs, owner = [], []
    for ci, (c, r) in enumerate(zip(claims, rankings)):
        for di in r[:topk]:
            pairs.append((c["claim"], corpus.text.iat[di])); owner.append(ci)
    P = verdict_probs(pairs)
    owner = np.array(owner)
    s = np.array([P[owner == i, 0].max() for i in range(len(claims))])
    r = np.array([P[owner == i, 1].max() for i in range(len(claims))])
    return s, r

def decide(s, r, tau):
    pred = np.where(s >= r, 0, 1)
    return np.where(np.maximum(s, r) < tau, 2, pred)

E2E = {name: claim_scores(claims, RANKS[name]["Hybrid"]) for name, claims in EVAL.items()}
taus = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
rows = []
for name, claims in EVAL.items():
    y = np.array([c["label"] for c in claims]); s, r = E2E[name]
    rows.append({"set": name, **{f"tau={t}": round(f1_score(y, decide(s, r, t), average="macro"), 3) for t in taus}})
print("End-to-end macro-F1 (retrieved evidence only):")
print(pd.DataFrame(rows).to_string(index=False))

**How to read this.** Step 2 scored HealthVer test at about 0.71 when it was *handed* the right abstract. These numbers include retrieval mistakes and the combine rule, so they should be lower. How much lower is the cost of retrieval.

In [ ]:
y = np.array([c["label"] for c in EVAL["HealthVer test"]]); s, r = E2E["HealthVer test"]
pred = decide(s, r, 0.5)
print("HealthVer test, tau=0.5 — what the system predicts (rows = true, cols = pred):")
print(pd.crosstab(pd.Series([LABELS[i] for i in y], name="true"), pd.Series([LABELS[i] for i in pred], name="pred")))

## 6. Save the pool and embeddings for the demo

In [ ]:
corpus.to_parquet(f"{OUT}/corpus.parquet", index=False)
np.save(emb_path, D)
json.dump(dict(emb_model=EMB_MODEL, query_prefix=QUERY_PREFIX, topk=TOPK, n_docs=len(corpus)),
          open(f"{OUT}/retrieval_config.json", "w"))
print(sorted(os.listdir(OUT)))

## What to check before Step 4
1. **Recall@5 per set (hybrid).** Below about 0.8 means retrieval is the bottleneck. Above 0.9 means the verdict model is.
2. **Hybrid vs BM25 vs Dense.** Hybrid should win or tie. If one method is far worse, drop it.
3. **End-to-end macro-F1 vs the step-2 number (0.71).** The gap is the price of finding evidence yourself.
4. **The crosstab**: lots of true SUPPORTS/REFUTES predicted as NEI means τ is too high or retrieval is missing the evidence.
5. **Reminder, say it in the pitch:** the pool is small (5,505 abstracts) and the verdict model saw many of these abstracts in training, with different claims. The system verifies claims about *this* literature, not the open web.